# Data Transformation (ETL)
**COSC 301 Project â€” Malaysia State-Level Socioeconomic & Health Outcomes**

This notebook reads raw data from `data/raw/`, applies cleaning rules identified in `02_explore.ipynb`, and writes two analysis-ready tables:

| Output table | Source files | Description |
|---|---|---|
| `health_state` | `moh_facilities`, `moh_beds`, `population_state` | Health infrastructure per capita by state (2019, 2022, 2024) |
| `combined_state` | `hh_income_parlimen`, `hh_poverty_parlimen`, `hh_income_state`, `hh_poverty_state` | Single analysis-ready table spanning 1970â€“2024; Gini available for 2019, 2022, 2024 only |

`socioeconomic_state` and `historical_state` are built as intermediate DataFrames and merged into `combined_state` â€” they are not written to disk.

All tables include a `territory_type` column classifying W.P. Kuala Lumpur (`capital`), W.P. Putrajaya (`admin`), W.P. Labuan (`island_ft`), and the 13 states (`state`).

Outputs go to `data/clean/*.csv` and `malaysia_project.db` (SQLite).

In [11]:
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = PROJECT_ROOT / "malaysia_project.db"

print("RAW_DIR :", RAW_DIR)
print("CLEAN_DIR:", CLEAN_DIR)
print("DB_PATH  :", DB_PATH)

RAW_DIR : c:\Users\aliff\Documents\GitHub\inequality-analysis\data\raw
CLEAN_DIR: c:\Users\aliff\Documents\GitHub\inequality-analysis\data\clean
DB_PATH  : c:\Users\aliff\Documents\GitHub\inequality-analysis\malaysia_project.db


In [12]:
# State name variants found in exploration â†’ canonical form
STATE_NORM = {
    "W.P Kuala Lumpur": "W.P. Kuala Lumpur",
    "W.P Labuan": "W.P. Labuan",
    "W.P Putrajaya": "W.P. Putrajaya",
    "WP Kuala Lumpur": "W.P. Kuala Lumpur",
    "WP Labuan": "W.P. Labuan",
    "WP Putrajaya": "W.P. Putrajaya",
}

# Official 3-letter state codes (used as primary key)
STATE_CODES = {
    "Johor": "JHR",
    "Kedah": "KDH",
    "Kelantan": "KTN",
    "Melaka": "MLK",
    "Negeri Sembilan": "NSN",
    "Pahang": "PHG",
    "Perak": "PRK",
    "Perlis": "PLS",
    "Pulau Pinang": "PNG",
    "Sabah": "SBH",
    "Sarawak": "SWK",
    "Selangor": "SGR",
    "Terengganu": "TRG",
    "W.P. Kuala Lumpur": "KUL",
    "W.P. Labuan": "LBN",
    "W.P. Putrajaya": "PJY",
}

# Federal territory classification for sensitivity analysis
FT_TYPES = {
    "W.P. Kuala Lumpur": "capital",
    "W.P. Putrajaya":    "admin",
    "W.P. Labuan":       "island_ft",
}

def normalise_state(series: pd.Series) -> pd.Series:
    return series.str.strip().replace(STATE_NORM)

## 1. Socioeconomic Table

**Sources:** `hh_income_parlimen.csv` and `hh_poverty_parlimen.csv`  
**Grain:** parliament constituency Ã— year  
**Target grain:** state Ã— year

**Aggregation rules:**
- `income_mean` / `income_median` â†’ unweighted mean across constituencies in the state  
- `poverty_absolute` â†’ unweighted mean across constituencies  
- `gini` â†’ Gini coefficient of constituency-level `income_mean` values within the state (measures inter-constituency income dispersion, not household-level inequality)

In [13]:
income_raw = pd.read_csv(RAW_DIR / "hh_income_parlimen.csv")
poverty_raw = pd.read_csv(RAW_DIR / "hh_poverty_parlimen.csv")

for df in [income_raw, poverty_raw]:
    df["state"] = normalise_state(df["state"])
    df["year"] = pd.to_datetime(df["date"]).dt.year

print("income : ", income_raw.shape, "| years:", sorted(income_raw.year.unique()))
print("poverty:", poverty_raw.shape, "| years:", sorted(poverty_raw.year.unique()))

income :  (666, 6) | years: [np.int32(2019), np.int32(2022), np.int32(2024)]
poverty: (666, 5) | years: [np.int32(2019), np.int32(2022), np.int32(2024)]


In [14]:
def gini_coefficient(series: pd.Series) -> float:
    """Gini coefficient of a 1-D array (population mean-based formula)."""
    arr = np.sort(series.dropna().values)
    n = len(arr)
    if n == 0 or arr.sum() == 0:
        return np.nan
    return (2 * np.dot(np.arange(1, n + 1), arr) / (n * arr.sum())) - (n + 1) / n

In [15]:
income_state = (
    income_raw
    .groupby(["state", "year"])
    .agg(
        income_mean=("income_mean", "mean"),
        income_median=("income_median", "mean"),
        gini=("income_mean", gini_coefficient),
        parlimen_count=("parlimen", "count"),
    )
    .reset_index()
)

poverty_state = (
    poverty_raw
    .groupby(["state", "year"])
    .agg(poverty_absolute=("poverty_absolute", "mean"))
    .reset_index()
)

socioeconomic = income_state.merge(poverty_state, on=["state", "year"])
socioeconomic["state_code"] = socioeconomic["state"].map(STATE_CODES)
socioeconomic["territory_type"] = socioeconomic["state"].map(FT_TYPES).fillna("state")

# Canonical column order
socioeconomic = socioeconomic[[
    "state_code", "state", "year", "territory_type",
    "income_mean", "income_median", "poverty_absolute", "gini",
    "parlimen_count",
]]

print("socioeconomic shape:", socioeconomic.shape)
socioeconomic.head()

socioeconomic shape: (48, 9)


,state_code,state,year,territory_type,income_mean,income_median,poverty_absolute,gini,parlimen_count
0,JHR,Johor,2019,state,7201.307692,5882.884615,4.623077,0.107621,26
1,JHR,Johor,2022,state,7529.730769,6215.346154,5.630769,0.108918,26
2,JHR,Johor,2024,state,8398.769231,7112.923077,3.092308,0.120739,26
3,KDH,Kedah,2019,state,5407.800000,4309.400000,8.540000,0.075098,15
4,KDH,Kedah,2022,state,5442.600000,4349.266667,8.346667,0.075074,15


In [16]:
# Sanity checks
assert socioeconomic["state_code"].notna().all(), "Unmapped states found"
assert not socioeconomic.duplicated(subset=["state", "year"]).any(), "Duplicate state-year keys"
assert socioeconomic["gini"].between(0, 1).all(), "Gini out of [0, 1] range"

missing = socioeconomic.isna().sum()
print("Missing values:\n", missing[missing > 0] if missing.any() else "  none")
print("\nStates Ã— years:", socioeconomic.groupby("year")["state"].count().to_dict())

Missing values:
   none

States Ã— years: {2019: 16, 2022: 16, 2024: 16}


## 2. Health Table

**Sources:** `moh_facilities.csv`, `moh_beds.csv`, `population_state.csv`

**Design:** MoH facilities and bed data are a **point-in-time snapshot** (no year column, assumed ~2024).
Infrastructure counts (hospitals, beds) are therefore **identical across all analysis years**.
However, the table is still keyed by `state x year` so it can be joined with the socioeconomic table.
The per-capita metrics (`beds_per_1000`, `facilities_per_100k`) **do vary by year** because the
population denominator is year-specific (DOSM intercensal estimates for 2019, 2022, 2024).

**Implication:** Do not interpret year-over-year changes in bed/facility counts as real trends --
they reflect population growth only. Cross-sectional comparisons between states are valid.

**Known data issue fixed here:** One facility in `moh_beds` had `beds_nonicu = -4`
(data entry error). Negative values are clamped to 0 before aggregation.

**Derived columns:**
- `beds_per_1000` = `beds_total` / (`population` / 1 000)
- `facilities_per_100k` = `facility_count` / (`population` / 100 000)

In [17]:
facilities_raw = pd.read_csv(RAW_DIR / "moh_facilities.csv")
beds_raw = pd.read_csv(RAW_DIR / "moh_beds.csv")
population_raw = pd.read_csv(RAW_DIR / "population_state.csv")

for df in [facilities_raw, beds_raw, population_raw]:
    df["state"] = normalise_state(df["state"])

print("facilities:", facilities_raw.shape)
print("beds      :", beds_raw.shape)
print("population:", population_raw.shape)

facilities: (3304, 9)
beds      : (149, 8)
population: (263679, 6)


In [18]:
# Population: keep overall totals only, convert thousands to actual count
pop = (
    population_raw[
        (population_raw["age"] == "overall")
        & (population_raw["sex"] == "both")
        & (population_raw["ethnicity"] == "overall")
    ]
    .copy()
)
pop["year"] = pd.to_datetime(pop["date"]).dt.year
pop["population"] = pop["population"] * 1_000
pop = pop[pop["year"].isin([2019, 2022, 2024])][["state", "year", "population"]]

print(
    "Population rows:",
    len(pop),
    "| States:",
    pop.state.nunique(),
    "| Years:",
    sorted(pop.year.unique()),
)
pop.head()

Population rows: 48 | States: 16 | Years: [np.int32(2019), np.int32(2022), np.int32(2024)]


,state,year,population
14622,Johor,2019,3761200.0
15819,Johor,2022,4028300.0
16617,Johor,2024,4184400.0
32037,Kedah,2019,2173700.0
33234,Kedah,2022,2163100.0


In [19]:
# Hospital count (type == 'hospital') and total facility count by state
hospital_count = (
    facilities_raw[facilities_raw.type == "hospital"]
    .groupby("state")
    .size()
    .rename("hospital_count")
    .reset_index()
)

facility_count = (
    facilities_raw
    .groupby("state")
    .size()
    .rename("facility_count")
    .reset_index()
)

print("Hospital count by state:")
print(hospital_count.sort_values("hospital_count", ascending=False).to_string(index=False))

Hospital count by state:
            state  hospital_count
            Sabah              25
          Sarawak              23
         Selangor              18
            Perak              16
           Pahang              13
            Johor              12
         Kelantan              10
            Kedah               9
  Negeri Sembilan               8
W.P. Kuala Lumpur               7
     Pulau Pinang               6
       Terengganu               6
           Melaka               4
   W.P. Putrajaya               2
           Perlis               1
      W.P. Labuan               1


In [20]:
# Beds: clamp negative values to 0 (one facility has beds_nonicu = -4, a known data entry error),
# then sum non-ICU and ICU by state.
beds_raw["beds_nonicu"] = beds_raw["beds_nonicu"].clip(lower=0)
beds_raw["beds_icu"] = beds_raw["beds_icu"].clip(lower=0)

negative_rows = beds_raw[
    (beds_raw["beds_nonicu"] < 0) | (beds_raw["beds_icu"] < 0)
]
if not negative_rows.empty:
    print("⚠ Rows with negative bed counts (clamped to 0):")
    print(negative_rows[["state", "beds_nonicu", "beds_icu"]])
else:
    print("✓ No negative bed counts after clamp")

beds_state = (
    beds_raw
    .groupby("state")
    .agg(
        beds_nonicu=("beds_nonicu", "sum"),
        beds_icu=("beds_icu", "sum"),
    )
    .reset_index()
)
beds_state["beds_total"] = beds_state["beds_nonicu"] + beds_state["beds_icu"]

print("
Beds by state:")
beds_state.sort_values("beds_total", ascending=False)

SyntaxError: unterminated string literal (detected at line 26) (2096188067.py, line 26)

In [ ]:
# Cross-join states with analysis years so the health table can be joined
# with the socioeconomic table on (state, year). Infrastructure counts
# (beds_total, facility_count) are identical across years -- only the per-capita
# denominators (population) differ. Do not interpret count changes as real trends.
states = sorted(pop["state"].dropna().unique())
ANALYSIS_YEARS = sorted(pop["year"].dropna().unique())

base = pd.DataFrame(
    [(s, y) for s in states for y in ANALYSIS_YEARS],
    columns=["state", "year"],
)

health = (
    base
    .merge(pop, on=["state", "year"], how="left")
    .merge(hospital_count, on="state", how="left")
    .merge(facility_count, on="state", how="left")
    .merge(beds_state, on="state", how="left")
)

health["beds_per_1000"] = health["beds_total"] / (health["population"] / 1_000)
health["facilities_per_100k"] = health["facility_count"] / (health["population"] / 100_000)
health["state_code"] = health["state"].map(STATE_CODES)
health["territory_type"] = health["state"].map(FT_TYPES).fillna("state")

health = health[[
    "state_code", "state", "year", "territory_type", "population",
    "hospital_count", "facility_count",
    "beds_nonicu", "beds_icu", "beds_total",
    "beds_per_1000", "facilities_per_100k",
]]

print("health shape:", health.shape)
health.head()

In [ ]:
assert health["state_code"].notna().all(), "Unmapped states"
assert not health.duplicated(subset=["state", "year"]).any(), "Duplicate state-year keys"

missing = health.isna().sum()
print("Missing values:\n", missing[missing > 0] if missing.any() else "  none")
min_beds = health["beds_per_1000"].min()
max_beds = health["beds_per_1000"].max()
print(f"\nbeds_per_1000 range: {min_beds:.3f} â€“ {max_beds:.3f}")

Missing values:
   none

beds_per_1000 range: 0.459 â€“ 7.618


## 3. Historical State Table

**Sources:** `hh_income_state.csv` and `hh_poverty_state.csv`  
**Grain:** state Ã— survey year (1970â€“2022)

This table uses DOSM's officially published state-level aggregates from the Household Income and Expenditure Survey (HIES). It covers all survey waves from 1970 to 2022 and complements the constituency-level socioeconomic table with a longer time series.

**Key differences from `socioeconomic_state`:**
- `income_mean` / `income_median` are DOSM's published state-level figures (population-weighted), not unweighted means of constituency values
- `poverty_absolute` is the official state poverty headcount ratio
- `poverty_hardcore` is the hardcore (extreme) poverty rate â€” available from 1984 onward, NaN for earlier waves
- `poverty_relative` is the relative poverty rate â€” available from 1989 onward, NaN for earlier waves
- No Gini â€” DOSM does not publish a state-level Gini via this API endpoint
- No `parlimen_count` â€” data is natively state-level

In [ ]:
income_hist = pd.read_csv(RAW_DIR / "hh_income_state.csv")
poverty_hist = pd.read_csv(RAW_DIR / "hh_poverty_state.csv")

for df in [income_hist, poverty_hist]:
    df["state"] = normalise_state(df["state"])
    df["year"] = pd.to_datetime(df["date"]).dt.year

print("income_hist :", income_hist.shape, "| years:", sorted(income_hist.year.unique()))
print("poverty_hist:", poverty_hist.shape, "| years:", sorted(poverty_hist.year.unique()))

income_hist : (303, 5) | years: [np.int32(1970), np.int32(1974), np.int32(1976), np.int32(1979), np.int32(1984), np.int32(1987), np.int32(1989), np.int32(1992), np.int32(1995), np.int32(1997), np.int32(1999), np.int32(2002), np.int32(2004), np.int32(2007), np.int32(2009), np.int32(2012), np.int32(2014), np.int32(2016), np.int32(2019), np.int32(2020), np.int32(2022)]
poverty_hist: (294, 6) | years: [np.int32(1970), np.int32(1976), np.int32(1979), np.int32(1984), np.int32(1987), np.int32(1989), np.int32(1992), np.int32(1995), np.int32(1997), np.int32(1999), np.int32(2002), np.int32(2004), np.int32(2007), np.int32(2009), np.int32(2012), np.int32(2014), np.int32(2016), np.int32(2019), np.int32(2020), np.int32(2022)]


In [ ]:
historical = income_hist[["state", "year", "income_mean", "income_median"]].merge(
    poverty_hist[["state", "year", "poverty_absolute", "poverty_hardcore", "poverty_relative"]],
    on=["state", "year"],
    how="left",
)

historical["state_code"] = historical["state"].map(STATE_CODES)
historical["territory_type"] = historical["state"].map(FT_TYPES).fillna("state")

historical = historical[[
    "state_code", "state", "year", "territory_type",
    "income_mean", "income_median",
    "poverty_absolute", "poverty_hardcore", "poverty_relative",
]]

# Sanity checks
assert historical["state_code"].notna().all(), "Unmapped states found"
assert not historical.duplicated(subset=["state", "year"]).any(), "Duplicate state-year keys"

missing = historical.isna().sum()
print("Missing values:")
print(missing[missing > 0].to_string() if missing.any() else "  none")
print(f"\nhistorical shape: {historical.shape} | years: {historical.year.min()}â€“{historical.year.max()}")
historical.tail()

Missing values:
income_median        11
poverty_absolute     12
poverty_hardcore     49
poverty_relative    105

historical shape: (303, 9) | years: 1970â€“2022


,state_code,state,year,territory_type,income_mean,income_median,poverty_absolute,poverty_hardcore,poverty_relative
298,PJY,W.P. Putrajaya,2014,admin,10401,7512.0,0.0,0.0,7.9
299,PJY,W.P. Putrajaya,2016,admin,11555,8275.0,0.0,0.0,6.8
300,PJY,W.P. Putrajaya,2019,admin,12840,9983.0,0.4,0.0,12.1
301,PJY,W.P. Putrajaya,2020,admin,12322,9743.0,0.2,0.0,11.2
302,PJY,W.P. Putrajaya,2022,admin,13473,10056.0,0.1,0.0,11.4


## 4. Write Outputs

Writing to:
- `data/clean/health_state.csv`
- `data/clean/combined_state.csv`
- `malaysia_project.db` (tables: `health_state`, `combined_state`)

`socioeconomic_state` and `historical_state` remain as in-memory intermediates only.

In [ ]:
health.to_csv(CLEAN_DIR / "health_state.csv", index=False)

print("âœ“ health_state.csv       :", len(health), "rows")

âœ“ health_state.csv       : 48 rows


In [ ]:
with sqlite3.connect(DB_PATH) as con:
    health.to_sql("health_state", con, if_exists="replace", index=False)

print("âœ“ Written to", DB_PATH.name)

# Verify round-trip
with sqlite3.connect(DB_PATH) as con:
    for table in ["health_state"]:
        n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table}", con)["n"][0]
        print(f"  {table}: {n} rows")

âœ“ Written to malaysia_project.db
  health_state: 48 rows


## 5. Combined State Table

Combines `historical_state` (1970â€“2016 + 2020) and `socioeconomic_state` (2019, 2022, 2024) into a single analysis-ready table.

**Design decisions:**
- **Overlap (2019, 2022):** rows from `socioeconomic_state` are used for these years because they carry income/poverty data at constituency resolution. `historical_state` rows for 2019 and 2022 are excluded to avoid duplicates.
- **Dropped columns:** `poverty_hardcore` and `poverty_relative` (sparse, not used in analysis); `parlimen_count` (administrative metadata).
- **Gini (1974â€“2022):** sourced from DOSM's [Household Income Inequality by State](https://open.dosm.gov.my/data-catalogue/hh_inequality_state) catalogue (`hh_inequality_state.csv`). These are official household-level Gini coefficients computed from HIES microdata, consistent across all survey waves.
- **Gini (2024):** no DOSM state-level publication yet; falls back to the inter-constituency income dispersion proxy derived from `hh_income_parlimen` (not comparable to the 1974â€“2022 series).
- **Gini (1970, 2020):** unavailable â€” not in `hh_inequality_state`; remains `NaN`.
- **Aggregation note:** income/poverty values for 2019 and 2022 use the constituency-derived figures (unweighted mean of constituency values), which differ slightly from DOSM's official state-level aggregates.

In [ ]:
# Load DOSM state-level Gini (1974â€“2022)
# Source: https://open.dosm.gov.my/data-catalogue/hh_inequality_state
gini_path = RAW_DIR / "hh_inequality_state.csv"

if gini_path.exists():
    gini_state_raw = pd.read_csv(gini_path)
else:
    print(f"Warning: {gini_path} not found. Using empty Gini table.")
    gini_state_raw = pd.DataFrame(columns=["date", "state", "gini"])
gini_state_raw["state"] = normalise_state(gini_state_raw["state"])
gini_state_raw["year"] = pd.to_datetime(gini_state_raw["date"]).dt.year
gini_lookup = gini_state_raw.set_index(["state", "year"])["gini"]

print("hh_inequality_state: ", gini_state_raw.shape,
      "| years:", sorted(gini_state_raw.year.unique()))

OVERLAP_YEARS = [2019, 2022]

hist_trimmed = (
    historical[~historical["year"].isin(OVERLAP_YEARS)]
    .drop(columns=["poverty_hardcore", "poverty_relative"])
    .assign(gini=np.nan)
)

socio_trimmed = socioeconomic.drop(columns=["parlimen_count"])

combined = (
    pd.concat([hist_trimmed, socio_trimmed], ignore_index=True)
    .sort_values(["state", "year"])
    .reset_index(drop=True)
)[["state_code", "state", "year", "territory_type",
   "income_mean", "income_median", "poverty_absolute", "gini"]]

# Apply DOSM household-level Gini for all years where it is available (1974â€“2022),
# overriding the inter-constituency proxy for 2019 and 2022.
# 2024 retains the inter-constituency value; 1970 and 2020 remain NaN.
dosm_mask = combined.set_index(["state", "year"]).index.isin(gini_lookup.index)
combined.loc[dosm_mask, "gini"] = combined.loc[dosm_mask].apply(
    lambda r: gini_lookup.get((r["state"], r["year"]), np.nan), axis=1
)

# Sanity checks
assert not combined.duplicated(subset=["state", "year"]).any(), "Duplicate state-year keys"
assert set(combined.columns) == {"state_code", "state", "year", "territory_type",
                                  "income_mean", "income_median", "poverty_absolute", "gini"}

print("\ncombined shape  :", combined.shape)
print("years covered   :", sorted(combined["year"].unique()))
print("gini non-null   :", combined["gini"].notna().sum(), "rows")
print("gini null by year:",
      combined.loc[combined["gini"].isna(), "year"].value_counts().to_dict())
missing = combined.isna().sum()
print("missing values  :")
print(missing[missing > 0].to_string() if (missing > 0).any() else "  none")

combined.to_csv(CLEAN_DIR / "combined_state.csv", index=False)

with sqlite3.connect(DB_PATH) as con:
    combined.to_sql("combined_state", con, if_exists="replace", index=False)

print("\nâœ“ combined_state.csv:", len(combined), "rows")
print("âœ“ Written to", DB_PATH.name)

combined.head(8)

hh_inequality_state:  (0, 4) | years: []

combined shape  : (319, 8)
years covered   : [np.int32(1970), np.int32(1974), np.int32(1976), np.int32(1979), np.int32(1984), np.int32(1987), np.int32(1989), np.int32(1992), np.int32(1995), np.int32(1997), np.int32(1999), np.int32(2002), np.int32(2004), np.int32(2007), np.int32(2009), np.int32(2012), np.int32(2014), np.int32(2016), np.int32(2019), np.int32(2020), np.int32(2022), np.int32(2024)]
gini non-null   : 48 rows
gini null by year: {2007: 16, 2009: 16, 2012: 16, 2014: 16, 2016: 16, 2020: 16, 1976: 14, 1984: 14, 1987: 14, 1989: 14, 1992: 14, 1995: 14, 1997: 14, 1999: 14, 2002: 14, 2004: 14, 1979: 13, 1970: 11, 1974: 11}
missing values  :
income_median        11
poverty_absolute     12
gini                271

âœ“ combined_state.csv: 319 rows
âœ“ Written to malaysia_project.db


,state_code,state,year,territory_type,income_mean,income_median,poverty_absolute,gini
0,JHR,Johor,1970,state,237.0,NaN,45.7,NaN
1,JHR,Johor,1974,state,382.0,269.0,NaN,NaN
2,JHR,Johor,1976,state,513.0,370.0,29.0,NaN
3,JHR,Johor,1979,state,731.0,518.0,18.2,NaN
4,JHR,Johor,1984,state,1065.0,793.0,12.2,NaN
5,JHR,Johor,1987,state,1060.0,816.0,11.1,NaN
6,JHR,Johor,1989,state,1150.0,880.0,10.1,NaN
7,JHR,Johor,1992,state,1713.0,1225.0,5.6,NaN
